# QM640 · 04 Jira test sets and full I1 baseline (RQ1)
Runs on the three Jira repositories extracted by notebook 02 (about 1.27 million issues). CPU only, no GPU units. Use *Runtime → Change runtime type → High-RAM* if available. About 20–40 minutes. *Runtime → Run all*.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
MAX_FEATURES = 100000   # TF-IDF vocabulary size
FIT_SAMPLE = 300000     # training issues used to learn the vocabulary

## 1. Settings
Same seed (640) and test-set size (385) as notebook 03. All outputs go to `Data/processed/jira_tests` and `Data/results`.

In [ ]:
import json, time, datetime, numpy as np, pandas as pd, pyarrow.parquet as pq
import matplotlib; import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.metrics import f1_score
from sklearn.preprocessing import normalize
from statsmodels.stats.proportion import proportion_confint
JIRA = DATA_ROOT/'processed'/'jira'; RESULTS = DATA_ROOT/'results'; OUTP = DATA_ROOT/'processed'/'jira_tests'
for p in (RESULTS/'figures', OUTP): p.mkdir(parents=True, exist_ok=True)
SEED, N_TEST, TOP_K = 640, 385, 10
rng = np.random.default_rng(SEED)

## 2. Load the three repositories and split by time
Text = summary + first 1,200 characters of the description. Issues dated before 2000 are excluded. Train = earliest 70%, validation = next 10%, test = latest 20%.

In [ ]:
COLS = ['repository','key','project','issuetype','priority','resolution','created','summary','description','components','issuelinks']
df = pd.concat([pd.read_parquet(f, columns=COLS) for f in sorted(JIRA.glob('*.parquet'))], ignore_index=True)
df['created'] = pd.to_datetime(df['created'], errors='coerce', utc=True)
df = df[df['created'] >= pd.Timestamp('2000-01-01', tz='UTC')].sort_values('created').reset_index(drop=True)   # 289 impossible dates removed
df['text'] = (df['summary'].fillna('') + '. ' + df['description'].fillna('').str.slice(0, 1200)).str.strip()
df['first_component'] = df['components'].map(lambda s: (json.loads(s) or [None])[0])
q70, q80 = df['created'].quantile(.7), df['created'].quantile(.8)
df['period'] = np.where(df['created'] < q70, 'train', np.where(df['created'] < q80, 'val', 'test'))
print(len(df), 'issues | train until', q70.date(), '| test from', q80.date()); print(df['period'].value_counts().to_dict())

## 3. TF-IDF text features
Words and word pairs, fitted on training-period issues only (a 300,000-issue sample, to stay within Colab memory). The project is added as a token.

In [ ]:
t0 = time.time()
vec = TfidfVectorizer(max_features=MAX_FEATURES, ngram_range=(1, 2), min_df=5, sublinear_tf=True, dtype=np.float32)
proj_tok = ' __proj_' + df['project'].fillna('NA').astype(str)
tr_all = np.where(df['period'] == 'train')[0]
fit_idx = rng.choice(tr_all, size=min(FIT_SAMPLE, len(tr_all)), replace=False)   # vocabulary from a training sample keeps memory within Colab limits
vec.fit(df['text'].values[fit_idx] + proj_tok.values[fit_idx])
X = vec.transform(df['text'] + proj_tok)
df = df.drop(columns=['summary', 'description']); import gc; gc.collect()
print('TF-IDF', X.shape, 'nnz', X.nnz, f'{time.time()-t0:.0f}s')

## 4. Classification decisions (I1)
D1 issue type and priority, D4 resolution (proxy for next action) and D5 project | component (proxy for team). Top classes plus *Other*. Majority-class baseline, naive Bayes and logistic regression; accuracy with Wilson 95% CI, macro-F1, risk–coverage at 98%/90% precision and ECE.

In [ ]:
def wilson(k, n): lo, hi = proportion_confint(k, n, method='wilson'); return f'{100*lo:.1f}–{100*hi:.1f}'
def coverage(conf, ok, p):
    o = np.argsort(-conf); prec = np.cumsum(ok[o]) / np.arange(1, len(o)+1); idx = np.where(prec >= p)[0]
    return round(100*(idx.max()+1)/len(o), 1) if len(idx) else 0.0
def ece(conf, ok, m=10):
    e = 0
    for a, b in zip(np.linspace(0, 1, m+1)[:-1], np.linspace(0, 1, m+1)[1:]):
        s = (conf > a) & (conf <= b)
        if s.any(): e += s.mean()*abs(ok[s].mean() - conf[s].mean())
    return round(e, 3)
TASKS = {'D1_issuetype': ('issuetype', 10), 'D1_priority': ('priority', 10), 'D4_resolution': ('resolution', 10), 'D5_component': ('proj_comp', 100)}
df['proj_comp'] = np.where(df['first_component'].notna(), df['project'].astype(str) + ' | ' + df['first_component'].astype(str), None)
rows, label_spaces = [], {}
for task, (col, top) in TASKS.items():
    lab = df[col]; has = lab.notna().values
    tr_idx = np.where(has & (df['period'] == 'train').values)[0]
    te_pool = np.where(has & (df['period'] == 'test').values)[0]
    classes = lab.iloc[tr_idx].value_counts().index[:top].tolist()
    y = lab.where(lab.isin(classes), 'Other')
    label_spaces[task] = classes + ['Other']
    te_idx = rng.choice(te_pool, size=N_TEST, replace=False)
    pd.DataFrame({'key': df['key'].values[te_idx], 'label': y.values[te_idx]}).to_csv(OUTP/f'testset_jira_{task}.csv', index=False)
    ytr, yte = y.values[tr_idx], y.values[te_idx]
    preds = {'Majority class': (np.array([pd.Series(ytr).value_counts().index[0]]*N_TEST), np.ones(N_TEST))}
    t0 = time.time(); nb = ComplementNB(alpha=.3).fit(X[tr_idx], ytr)
    preds['I1 TF-IDF + naive Bayes'] = (nb.predict(X[te_idx]), nb.predict_proba(X[te_idx]).max(1)); t_nb = time.time()-t0
    t0 = time.time(); sgd = SGDClassifier(loss='log_loss', alpha=2e-6, max_iter=15, tol=None, random_state=SEED, n_jobs=-1).fit(X[tr_idx], ytr)
    preds['I1 TF-IDF + logistic (SGD)'] = (sgd.predict(X[te_idx]), sgd.predict_proba(X[te_idx]).max(1)); t_lr = time.time()-t0
    for m, (p, conf) in preds.items():
        ok = (p == yte); k = int(ok.sum())
        rows.append({'Decision': task, 'Model': m, 'Train rows': len(tr_idx), 'Test rows': N_TEST, 'Classes': len(classes) + 1,
                     'Accuracy %': round(100*k/N_TEST, 1), '95% CI (Wilson)': wilson(k, N_TEST),
                     'Macro-F1': round(f1_score(yte, p, average='macro', zero_division=0), 3),
                     'Coverage % at 98% precision': coverage(conf, ok, .98) if m != 'Majority class' else None,
                     'Coverage % at 90% precision': coverage(conf, ok, .90) if m != 'Majority class' else None,
                     'ECE': ece(conf, ok) if m != 'Majority class' else None,
                     'Train+predict s': round({'Majority class': 0, 'I1 TF-IDF + naive Bayes': t_nb}.get(m, t_lr), 1)})
    pd.DataFrame({'key': df['key'].values[te_idx], 'label': yte, 'i1_pred': preds['I1 TF-IDF + logistic (SGD)'][0],
                  'i1_conf': preds['I1 TF-IDF + logistic (SGD)'][1]}).to_csv(OUTP/f'i1_predictions_{task}.csv', index=False)
    print(task, 'done')
cls = pd.DataFrame(rows); cls.to_csv(RESULTS/'rq1_jira_i1_full.csv', index=False); cls

## 5. Retrieval decisions (I1)
D2: find the earlier issue this one duplicates. D6: find the earlier issue that caused it. Only earlier issues from the same repository are searched, and a query is used only if its true answer exists earlier. Recall@1/5/10 and MRR@10.

In [ ]:
key_pos = pd.Series(np.arange(len(df)), index=df['key'].values)
created_ns = df['created'].values.astype('datetime64[ns]').astype(np.int64)
repo = df['repository'].values
def partners(links_json, kinds):
    out = []
    for l in json.loads(links_json):
        t, lab = (l.get('type') or '').lower(), (l.get('label') or '').lower()
        if any(k(t, lab) for k in kinds) and l.get('key') in key_pos.index: out.append(l['key'])
    return out
KINDS = {'D2_duplicate': [lambda t, l: 'duplicate' in t],
         'D6_cause': [lambda t, l: 'caused by' in l or (('cause' in t or 'incident' in t) and l.startswith('is caused'))]}
Xn = normalize(X).tocsr()
ret_rows = []
for task, kinds in KINDS.items():
    cand = []
    for i in np.where(df['period'].values == 'test')[0]:
        ps = [key_pos[k] for k in partners(df['issuelinks'].values[i], kinds)]
        ps = [p for p in ps if created_ns[p] < created_ns[i] and repo[p] == repo[i]]   # true answer must exist earlier
        if ps: cand.append((i, ps))
    n_avail = len(cand)
    sel = [cand[j] for j in rng.choice(len(cand), size=min(N_TEST, len(cand)), replace=False)]
    pd.DataFrame({'key': [df['key'].values[i] for i, _ in sel], 'answers': [json.dumps([df['key'].values[p] for p in ps]) for _, ps in sel]}).to_csv(OUTP/f'testset_jira_{task}.csv', index=False)
    hits = {1: 0, 5: 0, 10: 0}; rr = []; t0 = time.time(); topk_out = []
    for b in range(0, len(sel), 32):
        batch = sel[b:b+32]; Q = Xn[[i for i, _ in batch]]
        S = (Xn @ Q.T).toarray()                          # cosine similarity to every issue
        for j, (i, ps) in enumerate(batch):
            s = S[:, j].copy(); s[(created_ns >= created_ns[i]) | (repo != repo[i])] = -1   # earlier, same repository
            top = np.argpartition(-s, TOP_K)[:TOP_K]; top = top[np.argsort(-s[top])]
            ranks = [r+1 for r, t in enumerate(top) if t in ps]
            for k in hits: hits[k] += bool(ranks and ranks[0] <= k)
            rr.append(1/ranks[0] if ranks else 0); topk_out.append(json.dumps([df['key'].values[t] for t in top]))
    pd.DataFrame({'key': [df['key'].values[i] for i, _ in sel], 'i1_top10': topk_out}).to_csv(OUTP/f'i1_predictions_{task}.csv', index=False)
    n = len(sel)
    ret_rows.append({'Decision': task, 'Model': 'I1 TF-IDF cosine (lexical search)', 'Queries available': n_avail, 'Test rows': n,
                     'Recall@1 %': round(100*hits[1]/n, 1), 'Recall@5 %': round(100*hits[5]/n, 1), 'Recall@10 %': round(100*hits[10]/n, 1),
                     'Recall@10 95% CI': wilson(hits[10], n), 'MRR@10': round(float(np.mean(rr)), 3), 'Search s': round(time.time()-t0, 1)})
    print(task, 'done')
ret = pd.DataFrame(ret_rows); ret.to_csv(RESULTS/'rq1_jira_i1_retrieval.csv', index=False); ret

## 6. Save the ticket text for the language-model steps
One file with the text of every test ticket and every answer ticket, plus the allowed labels for each decision. Notebooks 05–08 read these files, so every instrument is scored on exactly the same tickets.

In [ ]:
keys = set()
for f in OUTP.glob('testset_jira_*.csv'):
    t = pd.read_csv(f); keys |= set(t['key'])
    if 'answers' in t: keys |= {k for a in t['answers'] for k in json.loads(a)}
tick = pd.concat([pd.read_parquet(f, columns=['repository','key','project','issuetype','priority','resolution','created','summary','description','components']) for f in sorted(JIRA.glob('*.parquet'))])
tick = tick[tick['key'].isin(keys)]
tick.to_parquet(OUTP/'jira_test_tickets.parquet', index=False)
json.dump(label_spaces, open(OUTP/'label_spaces.json', 'w'), indent=1)
print(len(tick), 'tickets saved for the language-model steps')

## 7. Summary file

In [ ]:
def mdt(d):
    try: return d.to_markdown(index=False)
    except Exception: return d.to_string()
lines = [f'# Step 04 summary - {datetime.datetime.now():%Y-%m-%d %H:%M}', '', f'Issues: {len(df):,}; split train < {q70.date()} <= val < {q80.date()} <= test',
         f'TF-IDF: {X.shape[1]:,} features', '', '## Classification (I1)', mdt(cls), '', '## Retrieval (I1)', mdt(ret), '', '## Label spaces', json.dumps(label_spaces)[:3000]]
(RESULTS/'step04_summary.md').write_text('\n'.join(lines)); print('\n'.join(lines[:30]))